In [ ]:
from pathlib import Path
import json
import functools
from typing import Any

import tensorflow as tf


def parse_proto(*, proto: tf.Tensor, meta: dict[str, Any]):
    # TODO, hand notes go through meta.json

    # Extract schemaless features
    empty_feature_container = {key: tf.io.VarLenFeature(tf.string) for key in meta["field_names"]}
    schemaless_features = tf.io.parse_single_example(proto, empty_feature_container)

    # Apply schema to features
    parsed_proto = {}
    for feature_name, schema in meta["features"].items():
        # decode the var length string tensor as the particular datatype recorded in
        # meta
        data = tf.io.decode_raw(
            schemaless_features[feature_name].values, getattr(tf, schema["dtype"])
        )

        # reshape flattened tensor into whatever shape is provided from meta
        data = tf.reshape(data, schema["shape"])

        if schema["type"] == "static":
            # data: (401, 1, 1)
            data = tf.tile(data, [meta["trajectory_length"], 1, 1])

        elif schema["type"] != "dynamic":
            raise ValueError(f"type not static or dynamic, is: {schema['type']}")

        parsed_proto[feature_name] = data

    return parsed_proto


def load_dataset(*, path: Path, split: str):

    with open(path / "meta.json", "r") as fp:
        metadata = json.loads(fp.read())

    lazy_dataset = tf.data.TFRecordDataset(str(path / f"{split}.tfrecord"))

    metadata_fused_parse = functools.partial(parse_proto, meta=metadata)

    lazy_dataset = lazy_dataset.map(metadata_fused_parse, num_parallel_calls=8)

    # optimize performance by prefetching the next batch while the current is being
    # consumed.
    lazy_dataset = lazy_dataset.prefetch(1)

    return lazy_dataset

In [ ]:
dataset_dir = Path("/home/jxn/dev/meshgraphnets/mgn/data/flag_simple")

ds = load_dataset(path=dataset_dir, split="train")